day 4: step 1:

in this step, Instead of loading one giant file all at once, we're going to split our data into weekly chunks, then use AutoLoader to feed those chunks one at a time, like they're new files arriving on different days, so we can practice building a pipeline that only processes what's new, not everything from scratch every time."

In [0]:
#DAY 4 - STEP 2:

from pyspark.sql import functions as F

orders_df = spark.read.csv(
    "/Volumes/workspace/default/raw_uploads/olist_orders_dataset.csv/",
    header = True,
    inferSchema = True,
)

orders_with_week = orders_df.withColumn(
    "order_week", F.date_trunc("week", "order_purchase_timestamp")

)
orders_with_week.select("order_id", "order_purchase_timestamp", "order_week").show(5)

#f.date_trunc("week", "order_purchase_timestamp") - takes precise timestamps and rounds down the start of that week. (truncate) here means to cut off the extra precision, keep only the bigger unit
#we do this bc we need a way to group thounsands of orders into batches by time, so we can simulate "week 1 dats arrives, then week 2 arrived" etc...

In [0]:
#DAY 4 - STEP 3:
distinct_weeks = [
    row["order_week"]
    for row in orders_with_week.select("order_week").distinct().collect()
]
distinct_weeks.sort()

print(f"Number of simulated batches: {len(distinct_weeks)}")
print(distinct_weeks[:5])

#.distinct() - a transformation that removes dulpicate values. we applied this to the orders_week column, so it givesa list of every unique week that appears in the data
#.collect() - pulls the actual result out of sparks distrubuted system and brings it into regular python. collects all the data from the dataframe to create a list
#row["order_week"] - this is a way to access the value of a specific column in a row of a dataframe. here we are accessing the value of the order_week column in each row of the dataframe
#

In [0]:
#DAY 4 - STEP 4:

DAILY_DROPS_PATH= "/Volumes/workspace/default/raw_uploads/simulated_drops/orders"

for week_start in distinct_weeks:
    batch_df = orders_with_week.filter(F.col("order_week") == week_start)
    batch_label = week_start.strftime("%Y_%m_%d")
    (
        batch_df.coalesce(1)
        .write.mode("overwrite")
        .options(header=True)
        .csv(f"{DAILY_DROPS_PATH}/batch_{batch_label}")
    )
print("done writing simulated weekly batches")

#the for loop is standard python repeating the write steps once per week
#.filter(F.col("order_week") == week_start) - this is a filter that selects only the rows where the order_week column matches the current week_start value
#.coalesce(1) - this is a transformation that combines multiple partitions into a single partition. here we are combining all the rows into a single partition
#.write.mode("overwrite") - this is a write mode that overwrites the existing data in the specified location

In [0]:
#DAY 4 - STEP 5:

BRONZE_TABLE_PATH = "/Volumes/workspace/default/raw_uploads/bronze/orders"
CHECKPOINT_PATH = "/Volumes/workspace/default/raw_uploads/_checkpoints/bronze_orders"

raw_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema")
    .option("header", "true")
    .load(f"{DAILY_DROPS_PATH}/*")
)

bronze_stream = raw_stream.withColumn(
    "_ingested_at", F.current_timestamp()
).withColumn(
    "_source_file", F.col("_metadata.file_path")
)

query = (
    bronze_stream.writeStream.format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/data")
    .trigger(availableNow=True)
    .start(BRONZE_TABLE_PATH)
)
query.awaitTermination()
print("Bronze Ingestion Complete.")

#spark.readStream - adding stream tells spark that we are setting up something that watches for data over time == fundemental switch from batch processing to incremental/streaming processing
#.format("cloudFiles") - this specifically activates autoloader
#.option("cloudFiles.schemaLocation", f") - this tells autoloader where to rememebr the data's column structure and stores the memory
#.option("header", "true") - same thing as before j written as a string
#.load(f"{DAILY_DROPS_PATH}/*") - the * is a ildcard meaning "look inside every subfolder"
#bronze_stream = raw_stream.withColumn("_ingested_at", F.current_timestamp()).withColumn("_source_file") - adds a new column recording the exsct moment each row was processed. the underscore prefix is a name convention signaling that this is metadata i personally added, not orgignal data
#f.col("_metadata.file_path")- spark automatically attached hidden metadaga to every row about while file it came frpm, so this pulls that out into its own column, so u can trace any row back to its og source
#.writeStream - the streaming equivalent of .write, matching the fact that we started with .readStream
#.option("checkpointlocation") - MOST IMPORTANT FOR AUTOLOADER'S CORE PROMISE!!! this is where spark permanently records "ive aready processed these specific files" without this, autoloader has no memory and would reprocess everything every time
#.trigger(availableNow=True) - this is a trigger that tells spark to process all the data that is currently available in the source location. this is a one-time trigger, so it will only process the data once and then stop.
#.start(BRONZE_TABLE_PATH) - this is the action that actually kicks off real processing, everything before was configurations
#query.awaitTermination() - this is a blocking call that waits for the stream to finish processing. it will block the notebook until the stream is finished.

In [0]:
# DAY 4 - STEP 6:

bronze_df = spark.read.format("delta").load(BRONZE_TABLE_PATH)
print(f"Bronze row count: {bronze_df.count()}")
bronze_df.select("order_id", "order_status", "_ingested_at", "_source_file").show(10, truncate=False)

In [0]:
#DAY4 - STEP 7:

# Re-run the exact same streaming cell from Step 5 again, with no new files added
query2 = (
    bronze_stream.writeStream.format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/data")
    .trigger(availableNow=True)
    .start(BRONZE_TABLE_PATH)
)
query2.awaitTermination()

bronze_df_recheck = spark.read.format("delta").load(BRONZE_TABLE_PATH)
print(f"Row count after re-run: {bronze_df_recheck.count()}")


#this is checking that the checkpont is working correcltvy
#idempotency - the ability to run the same operation multiple times without changing the result

In [0]:
#inspecting checkpoint folder:
display(dbutils.fs.ls(f"{CHECKPOINT_PATH}"))

In [0]:
# create one fake "new" batch that wasn't in your original set
import datetime

fake_new_batch = orders_with_week.filter(F.col("order_week") == distinct_weeks[0]).limit(5)
fake_new_batch.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{DAILY_DROPS_PATH}/batch_fake_new_arrival"
)

# run ingestion again
query3 = (
    bronze_stream.writeStream.format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/data")
    .trigger(availableNow=True)
    .start(BRONZE_TABLE_PATH)
)
query3.awaitTermination()

bronze_df_after_new_file = spark.read.format("delta").load(BRONZE_TABLE_PATH)
print(f"Row count after adding ONE new file: {bronze_df_after_new_file.count()}")

In [0]:
#BRONZE TABLE actual metadata columns
bronze_df.groupBy("_source_file").count().orderBy(F.desc("count")).show(20, truncate=False)

i simulated incremental file arrival by splitting the static olist dataset into weekly batches, since auto loader's core value( only processing new files) cannot be demonstrated agiainst a single static file load. i verified idempotency by re-running ingestion with no new files (zero rows added) and by adding a single new file( exactly that files rows were added, nothing else reprocessed)"

In [0]:
import requests

response = requests.get("https://date.nager.at/api/v3/PublicHolidays/2018/BR")
print(response.status_code)
print(response.json()[:2])
# we are testing whether the outbound internest works from this notebook

In [0]:
import json
from pyspark.sql import functions as F

def fetch_holidays(country_code:str, year:int) -> list[dict]:
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

all_holidays = []
for yr in [2016, 2017, 2018]:
    all_holidays.extend(fetch_holidays("BR", yr))

print(f"Pulled {len(all_holidays)}holiday records")
print(json.dumps(all_holidays[:3], indent=2))

#def fetch_holidays... - a plain python function, wrapping this logic in a function rahter than writing it inline, makes it reusable
#responseraise_for_status - a requests library that checks if the api call was successful, if not, it raises an exception immediately, rather than waiting for the next line of code
#.extend vs .append - extend adds each item from a list individually into all_holidays, while append would added the entires years list as one nested item. we want one flat list of holidays across all years, so extend is correct here
#json.dumps(..., indent=2) a nicely formatted print of the raw api repsponse, so you can actualy read the structure before turning it into a dataframe


In [0]:
#simplifly each holiday record down to just the fieled we need,
#avoiding fields like "countires" that have inconsistent types across records

cleaned_holidays = [
    {
        "date": h["date"],
        "localName": h["localName"],
        "name": h["name"],
        "countryCode": h["countryCode"],
        "fixed": h["fixed"],
    }
    for h in all_holidays
]
#spark.createDataFrame - instead of reading from a file, this converts a plain python list(api reponse) directly into a spark data frame
holidays_df = spark.createDataFrame (cleaned_holidays)
holidays_df = holidays_df.withColumn(
    "ingested_at", F.current_timestamp()
).withColumn(
    "source", F.lit("nager_date_api")
)
#F.lit("nager_date_api") - new function: lit - "literal." normally withColumn calculations reference existing column values, f.lit is for when you want to inset a fixed, constant alue in every row
holidays_df.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/bronze/holidays"
)

print("Holiday reference table written to Bronze.")
holidays_df.show(10, truncate=False)

#[{...} for h in all_holidays] - another list comprehension (seen this pettern before, back on day 4, with distinct_weeks) in englihs: for every holiday record h in the original list, build a new, smaller dictionary containing only these 5 feidls, and collect all of those into a new list
#why this fixes the error: date, local name, name, and countrycode and fixed are all simple, consistent types( text and boolean) accorss every record, no nested lists, no nulls mixed with lists. spark can confidently infer these

In [0]:
orders_bronze = spark.read.format("delta").load("/Volumes/workspace/default/raw_uploads/bronze/orders")
holidays_bronze = spark.read.format("delta").load("/Volumes/workspace/default/raw_uploads/bronze/holidays")

print(f"Orders Bronze row count: {orders_bronze.count()}")
print(f"Holidays Bronze row count: {holidays_bronze.count()}")

orders_bronze.printSchema()
holidays_bronze.printSchema()

#this is a "State of the Union" check, confirming both tables you built this week actually exist, have sensible row counts, and have the schema you expect, all in one place. Worth doing this any time you pick a project back up after a gap

In [0]:
customers_df = spark.read.csv(
    "/Volumes/workspace/default/raw_uploads/olist_customers_dataset.csv",
    header=True,
    inferSchema=True,
)

# Check 1: orders referencing a customer_id that doesn't exist in customers
orphaned_orders = orders_bronze.join(customers_df, on="customer_id", how="left_anti")
print(f"Orders with no matching customer: {orphaned_orders.count()}")

# Check 2: any duplicate order_id values (should be unique)
from pyspark.sql import functions as F

duplicate_check = (
    orders_bronze.groupBy("order_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
)
print(f"Duplicate order_ids found: {duplicate_check.count()}")

# Check 3: any orders with a null order_status
null_status_check = orders_bronze.filter(F.col("order_status").isNull())
print(f"Orders with null status: {null_status_check.count()}")

#this matters 